# k-Sensitivity Analysis — Fairlet Only
Supplement to main k-sensitivity notebook.
Fairlet is subsampled to n=5000 for datasets > 10,000 samples to manage runtime.
Results are merged into the main k_sensitivity.pkl after completion.


## Cell 1: Imports + Setup (copy from main notebook)

In [3]:
import numpy as np
import pandas as pd
import pickle
import requests
import io
from scipy.io import arff
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    silhouette_score, silhouette_samples, calinski_harabasz_score,
    davies_bouldin_score, adjusted_rand_score, normalized_mutual_info_score,
    adjusted_mutual_info_score, fowlkes_mallows_score, v_measure_score
)
from scipy.spatial.distance import cdist, pdist
from scipy.stats import entropy
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['axes.spines.top']   = False
matplotlib.rcParams['axes.spines.right'] = False
import seaborn as sns
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')
np.random.seed(0)

## Cell 2: Original 5 Dataset Loaders

In [5]:
# =============================================================================
# SECTION 1: ORIGINAL 5 DATASET LOADERS (unchanged)
# =============================================================================

def load_adult():
    print("\n" + "="*80)
    print("Loading Adult Income Dataset")
    print("="*80)
    columns = [
        'age', 'workclass', 'fnlwgt', 'education', 'education-num',
        'marital-status', 'occupation', 'relationship', 'race', 'sex',
        'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
    ]
    adult = pd.read_csv("C:/Users/G1237/OneDrive/Desktop/Mphil/K-mean_fairness/Assessing Fairness-Utility Trade‑offs in K-Means Clustering- A Dual‑Strategy Evaluation Framework/data/adult/adult.data", names=columns, header=None)
    adult = adult.replace(' ?', np.nan).dropna()
    adult['sex_binary'] = (adult['sex'].str.strip() == 'Male').astype(int)
    adult_features = [c for c in adult.columns if c not in ["income", "sex", "race", "sex_binary"]]
    print(f"Shape: {adult.shape}")
    print(f"Gender: 0(F)={( adult['sex_binary']==0).sum()}, 1(M)={(adult['sex_binary']==1).sum()}")
    return adult, adult_features, 'sex_binary'


def load_compas():
    print("\n" + "="*80)
    print("Loading COMPAS Dataset")
    print("="*80)
    compas = pd.read_csv("C:/Users/G1237/OneDrive/Desktop/Mphil/K-mean_fairness/Assessing Fairness-Utility Trade‑offs in K-Means Clustering- A Dual‑Strategy Evaluation Framework/data/compas/compas-scores.csv")
    compas = compas[
        (compas['days_b_screening_arrest'] <= 30) &
        (compas['days_b_screening_arrest'] >= -30) &
        (compas['is_recid'] != -1)
    ]
    selected_cols = compas.columns[compas.isnull().sum() == 0].tolist()
    compas = compas[selected_cols]
    compas['sex_binary'] = (compas['sex'] == 'Male').astype(int)
    compas_features = [c for c in compas.columns if c not in ["is_recid", "sex", "race", "sex_binary"]]
    print(f"Shape: {compas.shape}")
    print(f"Gender: 0(F)={(compas['sex_binary']==0).sum()}, 1(M)={(compas['sex_binary']==1).sum()}")
    return compas, compas_features, 'sex_binary'


def load_german():
    print("\n" + "="*80)
    print("Loading German Credit Dataset")
    print("="*80)
    german_columns = [
        'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
        'savings_status', 'employment', 'installment_rate', 'personal_status_sex',
        'other_parties', 'residence_since', 'property_magnitude', 'age',
        'other_payment_plans', 'housing', 'existing_credits', 'job',
        'num_dependents', 'own_telephone', 'foreign_worker', 'class'
    ]
    german = pd.read_csv("C:/Users/G1237/OneDrive/Desktop/Mphil/K-mean_fairness/Assessing Fairness-Utility Trade‑offs in K-Means Clustering- A Dual‑Strategy Evaluation Framework/data/german/german.data", sep=' ', names=german_columns, header=None)
    def extract_sex(s):
        return 'male' if s in ['A91', 'A93', 'A94'] else ('female' if s in ['A92', 'A95'] else 'unknown')
    german['sex'] = german['personal_status_sex'].apply(extract_sex)
    german = german[german['sex'] != 'unknown']
    german['sex_binary'] = (german['sex'] == 'male').astype(int)
    german_features = [c for c in german.columns if c not in ["class", "sex", "sex_binary"]]
    print(f"Shape: {german.shape}")
    print(f"Gender: 0(F)={(german['sex_binary']==0).sum()}, 1(M)={(german['sex_binary']==1).sum()}")
    return german, german_features, 'sex_binary'


def load_credit():
    print("\n" + "="*80)
    print("Loading Default Credit Card Dataset")
    print("="*80)
    credit = pd.read_csv("C:/Users/G1237/OneDrive/Desktop/Mphil/K-mean_fairness/Assessing Fairness-Utility Trade‑offs in K-Means Clustering- A Dual‑Strategy Evaluation Framework/data/default of credit card clients/default of credit card clients.csv", header=0)
    if 'X1' in credit.columns or credit.columns[0] == 'ID':
        credit.columns = [
            'ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
            'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
            'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
            'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default'
        ]
    credit = credit.drop(columns=['ID'])
    credit['SEX'] = pd.to_numeric(credit['SEX'], errors='coerce')
    credit['sex'] = credit['SEX'].map({1.0: 'male', 2.0: 'female'})
    credit = credit.dropna(subset=['sex'])
    credit['sex_binary'] = (credit['sex'] == 'male').astype(int)
    credit_features = [c for c in credit.columns if c not in ["default", "sex", "SEX", "sex_binary"]]
    print(f"Shape: {credit.shape}")
    print(f"Gender: 0(F)={(credit['sex_binary']==0).sum()}, 1(M)={(credit['sex_binary']==1).sum()}")
    return credit, credit_features, 'sex_binary'


def load_law():
    print("\n" + "="*80)
    print("Loading Law School (LSAC) Dataset")
    print("="*80)
    lsac = pd.read_csv("C:/Users/G1237/OneDrive/Desktop/Mphil/K-mean_fairness/Assessing Fairness-Utility Trade‑offs in K-Means Clustering- A Dual‑Strategy Evaluation Framework/data/law/law_dataset.csv")
    lsac['sex'] = lsac['male'].map({1: 'Male', 0: 'Female', 1.0: 'Male', 0.0: 'Female'})
    lsac = lsac[lsac['sex'].notna()]
    lsac['sex_binary'] = (lsac['sex'] == 'Male').astype(int)
    drop_cols = ['male', 'sex', 'pass_bar', 'sex_binary']
    lsac_features = [c for c in lsac.columns if c not in drop_cols]
    print(f"Shape: {lsac.shape}")
    print(f"Gender: 0(F)={(lsac['sex_binary']==0).sum()}, 1(M)={(lsac['sex_binary']==1).sum()}")
    return lsac, lsac_features, 'sex_binary'

# =============================================================================
# RACE & COMBINED LOADERS — Adult, COMPAS, Law
# German, Credit: gender only
# =============================================================================

def load_adult_race():
    columns = [
        'age', 'workclass', 'fnlwgt', 'education', 'education-num',
        'marital-status', 'occupation', 'relationship', 'race', 'sex',
        'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
    ]
    df = pd.read_csv("C:/Users/G1237/OneDrive/Desktop/Mphil/K-mean_fairness/Assessing Fairness-Utility Trade‑offs in K-Means Clustering- A Dual‑Strategy Evaluation Framework/data/adult/adult.data", names=columns, header=None)
    df = df.replace(' ?', np.nan).dropna()
    df['race'] = df['race'].str.strip()
    df['race_binary'] = (df['race'] == 'White').astype(int)
    features = [c for c in df.columns if c not in ["income", "sex", "race", "race_binary"]]
    print(f"Adult (race) | 0(NonWhite)={(df['race_binary']==0).sum()}, 1(White)={(df['race_binary']==1).sum()}")
    return df, features, 'race_binary'

def load_adult_combined():
    columns = [
        'age', 'workclass', 'fnlwgt', 'education', 'education-num',
        'marital-status', 'occupation', 'relationship', 'race', 'sex',
        'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
    ]
    df = pd.read_csv("C:/Users/G1237/OneDrive/Desktop/Mphil/K-mean_fairness/Assessing Fairness-Utility Trade‑offs in K-Means Clustering- A Dual‑Strategy Evaluation Framework/data/adult/adult.data", names=columns, header=None)
    df = df.replace(' ?', np.nan).dropna()
    df['race'] = df['race'].str.strip()
    df['sex_binary']  = (df['sex'].str.strip() == 'Male').astype(int)
    df['race_binary'] = (df['race'] == 'White').astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']
    features = [c for c in df.columns if c not in ["income","sex","race","sex_binary","race_binary","combined"]]
    print(f"Adult (combined):\n{df['combined'].value_counts().sort_index()}")
    print("0=White-Female, 1=White-Male, 2=NonWhite-Female, 3=NonWhite-Male")
    return df, features, 'combined'

def load_compas_race():
    df = pd.read_csv("C:/Users/G1237/OneDrive/Desktop/Mphil/K-mean_fairness/Assessing Fairness-Utility Trade‑offs in K-Means Clustering- A Dual‑Strategy Evaluation Framework/data/compas/compas-scores.csv")
    df = df[(df['days_b_screening_arrest']<=30)&(df['days_b_screening_arrest']>=-30)&(df['is_recid']!=-1)]
    df = df[df.columns[df.isnull().sum()==0].tolist()]
    df = df[df['race'].isin(['Caucasian','African-American'])].copy()
    df['race_binary'] = (df['race'] == 'African-American').astype(int)
    features = [c for c in df.columns if c not in ["is_recid","sex","race","race_binary"]]
    print(f"COMPAS (race) | 0(White)={(df['race_binary']==0).sum()}, 1(Black)={(df['race_binary']==1).sum()}")
    return df, features, 'race_binary'

def load_compas_combined():
    df = pd.read_csv("C:/Users/G1237/OneDrive/Desktop/Mphil/K-mean_fairness/Assessing Fairness-Utility Trade‑offs in K-Means Clustering- A Dual‑Strategy Evaluation Framework/data/compas/compas-scores.csv")
    df = df[(df['days_b_screening_arrest']<=30)&(df['days_b_screening_arrest']>=-30)&(df['is_recid']!=-1)]
    df = df[df.columns[df.isnull().sum()==0].tolist()]
    df = df[df['race'].isin(['Caucasian','African-American'])].copy()
    df['sex_binary']  = (df['sex'] == 'Male').astype(int)
    df['race_binary'] = (df['race'] == 'African-American').astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']
    features = [c for c in df.columns if c not in ["is_recid","sex","race","sex_binary","race_binary","combined"]]
    print(f"COMPAS (combined):\n{df['combined'].value_counts().sort_index()}")
    print("0=White-Female, 1=White-Male, 2=Black-Female, 3=Black-Male")
    return df, features, 'combined'

def load_law_race():
    df = pd.read_csv("C:/Users/G1237/OneDrive/Desktop/Mphil/K-mean_fairness/Assessing Fairness-Utility Trade‑offs in K-Means Clustering- A Dual‑Strategy Evaluation Framework/data/law/law_dataset.csv")
    df['sex'] = df['male'].map({1:'Male',0:'Female',1.0:'Male',0.0:'Female'})
    df = df[df['sex'].notna()]
    df['race_binary'] = df['racetxt'].astype(int)  # 1=White, 0=NonWhite
    features = [c for c in df.columns if c not in ['male','sex','pass_bar','racetxt','race_binary']]
    print(f"Law (race) | 0(NonWhite)={(df['race_binary']==0).sum()}, 1(White)={(df['race_binary']==1).sum()}")
    return df, features, 'race_binary'

def load_law_combined():
    df = pd.read_csv("C:/Users/G1237/OneDrive/Desktop/Mphil/K-mean_fairness/Assessing Fairness-Utility Trade‑offs in K-Means Clustering- A Dual‑Strategy Evaluation Framework/data/law/law_dataset.csv")
    df['sex'] = df['male'].map({1:'Male',0:'Female',1.0:'Male',0.0:'Female'})
    df = df[df['sex'].notna()]
    df['sex_binary']  = (df['sex'] == 'Male').astype(int)
    df['race_binary'] = df['racetxt'].astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']
    features = [c for c in df.columns if c not in ['male','sex','pass_bar','racetxt','sex_binary','race_binary','combined']]
    print(f"Law (combined):\n{df['combined'].value_counts().sort_index()}")
    print("0=White-Female, 1=White-Male, 2=NonWhite-Female, 3=NonWhite-Male")
    return df, features, 'combined'

## Cell 3: New Dataset Loaders (D6-D8)

In [7]:
# SECTION 2: NEW DATASET LOADERS — D6, D7, D8
# Each supports mode='gender' | 'race' | 'combined'
# =============================================================================

def load_diabetes(mode='gender'):
    """
    D6: Diabetes 130-US Hospitals (UCI id=296)
    mode: 'gender' | 'race' | 'combined'
    race: 1=Caucasian, 0=AfricanAmerican
    combined: 0=White-Female, 1=White-Male, 2=Black-Female, 3=Black-Male
    """
    print("\n" + "="*80)
    print(f"Loading Diabetes 130-US Dataset (mode={mode})")
    print("="*80)
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=296)
    df = ds.data.features.copy()

    drop_cols = ['encounter_id', 'patient_nbr', 'weight', 'payer_code', 'medical_specialty']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df = df[df['gender'].isin(['Male', 'Female'])]
    df = df[df['race'] != '?'].dropna(subset=['gender', 'race'])

    df['sex_binary'] = (df['gender'] == 'Male').astype(int)

    # For race/combined: keep only Caucasian and AfricanAmerican
    df_race = df[df['race'].isin(['Caucasian', 'AfricanAmerican'])].copy()
    df_race['race_binary'] = (df_race['race'] == 'Caucasian').astype(int)
    df_race['combined'] = df_race['race_binary'] * 2 + df_race['sex_binary']

    base_excl = ['gender', 'race', 'sex_binary', 'race_binary', 'combined']

    if mode == 'gender':
        features = [c for c in df.columns if c not in base_excl]
        print(f"Shape: {df.shape} | Gender: 0(F)={(df['sex_binary']==0).sum()}, 1(M)={(df['sex_binary']==1).sum()}")
        return df, features, 'sex_binary'
    elif mode == 'race':
        features = [c for c in df_race.columns if c not in base_excl]
        print(f"Shape: {df_race.shape} | Race: 0(Black)={(df_race['race_binary']==0).sum()}, 1(White)={(df_race['race_binary']==1).sum()}")
        return df_race, features, 'race_binary'
    elif mode == 'combined':
        features = [c for c in df_race.columns if c not in base_excl]
        print(f"Shape: {df_race.shape} | Combined:\n{df_race['combined'].value_counts().sort_index()}")
        print("0=White-Female, 1=White-Male, 2=Black-Female, 3=Black-Male")
        return df_race, features, 'combined'
    else:
        raise ValueError(f"mode must be 'gender'/'race'/'combined', got '{mode}'")


def load_dutch(mode='gender'):
    """
    D7: Dutch Census 2001
    mode: 'gender' only (no race attribute in this dataset)
    sex: 1=male, 2=female → sex_binary: 1=male, 0=female
    """
    print("\n" + "="*80)
    print(f"Loading Dutch Census Dataset (mode={mode})")
    print("="*80)
    if mode != 'gender':
        raise ValueError("Dutch Census only supports mode='gender' (no race attribute)")

    url = ('https://raw.githubusercontent.com/tailequy/fairness_dataset'
           '/main/Dutch_census/dutch_census_2001.arff')
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data, meta = arff.loadarff(io.StringIO(resp.text))
        df = pd.DataFrame(data)
        for col in df.select_dtypes(['object']).columns:
            df[col] = df[col].str.decode('utf-8')
        print(f"Loaded from URL. Shape: {df.shape}")
    except Exception as e:
        print(f"URL failed ({e}), trying local file ./data/dutch/dutch_census_2001.csv")
        df = pd.read_csv('./data/dutch/dutch_census_2001.csv')

    sex_col = [c for c in df.columns if c.lower() == 'sex'][0]
    unique_vals = set(df[sex_col].astype(str).unique())
    if unique_vals.issubset({'1', '2'}):
        df['sex_binary'] = (df[sex_col].astype(str) == '1').astype(int)
    else:
        df['sex_binary'] = (df[sex_col].str.lower().isin(['male', 'm', '1'])).astype(int)

    target_col = next((c for c in df.columns if c.lower() in ['occupation', 'label', 'class']), None)
    excl = [sex_col, 'sex_binary'] + ([target_col] if target_col else [])
    features = [c for c in df.columns if c not in excl]

    print(f"Gender: 0(F)={(df['sex_binary']==0).sum()}, 1(M)={(df['sex_binary']==1).sum()}")
    return df, features, 'sex_binary'


def load_meps(mode='gender'):
    print("\n" + "="*80)
    print(f"Loading MEPS Panel 20 (2016) Dataset (mode={mode})")
    print("="*80)

    meps_dir = r'C:\Users\G1237\anaconda3\Lib\site-packages\aif360\data\raw\meps'
    df = pd.read_csv(meps_dir + r'\h192.csv', low_memory=False)
    print(f"Loaded h192.csv. Shape: {df.shape}")

    keep_cols = ['AGE31X','AGE42X','AGE53X','SEX','RACEV1X','MARRY31X','MARRY42X','MARRY53X',
                 'EDUCYR','HIDEG','FTSTU31X','FTSTU42X','FTSTU53X','ACTDTY31','ACTDTY42','ACTDTY53',
                 'HONRDC31','HONRDC42','HONRDC53','WRGFD31X','WRGFD42X','WRGFD53X',
                 'TTLP16X','POVCAT16','INSCOV16']
    available = [c for c in keep_cols if c in df.columns]
    print(f"Available key columns: {len(available)}/{len(keep_cols)}")
    df = df[available].dropna()
    print(f"Shape after dropna: {df.shape}")

    df['sex_binary']  = (df['SEX'] == 1).astype(int)
    df['race_binary'] = (df['RACEV1X'] == 1).astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']

    print(f"Sex distribution: {df['SEX'].value_counts().to_dict()}")
    print(f"Race distribution: {df['RACEV1X'].value_counts().to_dict()}")

    base_excl = ['SEX', 'RACEV1X', 'sex_binary', 'race_binary', 'combined']
    features = [c for c in df.columns if c not in base_excl]

    if mode == 'gender':
        print(f"Gender: 0(F)={(df['sex_binary']==0).sum()}, 1(M)={(df['sex_binary']==1).sum()}")
        return df, features, 'sex_binary'
    elif mode == 'race':
        print(f"Race: 0(NonWhite)={(df['race_binary']==0).sum()}, 1(White)={(df['race_binary']==1).sum()}")
        return df, features, 'race_binary'
    elif mode == 'combined':
        print(f"Combined:\n{df['combined'].value_counts().sort_index()}")
        print("0=White-Female, 1=White-Male, 2=NonWhite-Female, 3=NonWhite-Male")
        return df, features, 'combined'
    else:
        raise ValueError(f"mode must be gender/race/combined, got '{mode}'")

## Cell 4: Preprocessing

In [9]:
# =============================================================================
# SECTION 3: PREPROCESSING
# =============================================================================

def preprocess_dataset(df, feature_cols, sensitive_col):
    X_df = df[feature_cols].copy()
    for col in X_df.select_dtypes(include=['object']).columns:
        X_df[col] = LabelEncoder().fit_transform(X_df[col].astype(str))
    X = X_df.values.astype(float)
    if np.any(np.isnan(X)) or np.any(np.isinf(X)):
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    X = StandardScaler().fit_transform(X)
    sensitive_attr = df[sensitive_col].values.astype(int)
    print(f"  Final shape: {X.shape}")
    print(f"  Sensitive groups: {dict(zip(*np.unique(sensitive_attr, return_counts=True)))}")
    return X, sensitive_attr



## Cell 5: Load All Datasets

In [11]:
# =============================================================================
# LOAD ALL DATASETS — Part A (13 configs) + Part B (8 configs) = 21 total
# =============================================================================

print("\n" + "="*80)
print("LOADING ALL DATASETS")
print("="*80)
datasets = {}

# ── Part A: Original 5 datasets (gender + race + combined where applicable) ──
for name, loader in [
    ('adult',   load_adult),
    ('compas',  load_compas),
    ('german',  load_german),
    ('credit',  load_credit),
    ('law',     load_law),
]:
    try:
        df, features, sc = loader()
        X, S = preprocess_dataset(df, features, sc)
        datasets[name] = {'X': X, 'sensitive': S}
        print(f"✓ {name}: {X.shape}")
    except Exception as e:
        print(f"✗ {name}: {e}")

for name, loader in [
    ('adult_race',      load_adult_race),
    ('adult_combined',  load_adult_combined),
    ('compas_race',     load_compas_race),
    ('compas_combined', load_compas_combined),
    ('law_race',        load_law_race),
    ('law_combined',    load_law_combined),
]:
    try:
        df, features, sc = loader()
        X, S = preprocess_dataset(df, features, sc)
        datasets[name] = {'X': X, 'sensitive': S}
        print(f"✓ {name}: {X.shape}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# ── Part B: New datasets ──────────────────────────────────────────────────────
for mode in ['gender', 'race', 'combined']:
    name = f'diabetes_{mode}'
    try:
        df, features, sc = load_diabetes(mode=mode)
        X, S = preprocess_dataset(df, features, sc)
        datasets[name] = {'X': X, 'sensitive': S}
        print(f"✓ {name}: {X.shape}")
    except Exception as e:
        print(f"✗ {name}: {e}")

try:
    df, features, sc = load_dutch(mode='gender')
    X, S = preprocess_dataset(df, features, sc)
    datasets['dutch_gender'] = {'X': X, 'sensitive': S}
    print(f"✓ dutch_gender: {X.shape}")
except Exception as e:
    print(f"✗ dutch_gender: {e}")

for mode in ['gender', 'race', 'combined']:
    name = f'meps_{mode}'
    try:
        df, features, sc = load_meps(mode=mode)
        X, S = preprocess_dataset(df, features, sc)
        datasets[name] = {'X': X, 'sensitive': S}
        print(f"✓ {name}: {X.shape}")
    except Exception as e:
        print(f"✗ {name}: {e}")

dataset_names = list(datasets.keys())
print(f"\nTotal: {len(dataset_names)} configurations loaded")



LOADING ALL DATASETS

Loading Adult Income Dataset
Shape: (30162, 16)
Gender: 0(F)=9782, 1(M)=20380
  Final shape: (30162, 12)
  Sensitive groups: {0: 9782, 1: 20380}
✓ adult: (30162, 12)

Loading COMPAS Dataset
Shape: (9395, 31)
Gender: 0(F)=1933, 1(M)=7462
  Final shape: (9395, 27)
  Sensitive groups: {0: 1933, 1: 7462}
✓ compas: (9395, 27)

Loading German Credit Dataset
Shape: (1000, 23)
Gender: 0(F)=310, 1(M)=690
  Final shape: (1000, 20)
  Sensitive groups: {0: 310, 1: 690}
✓ german: (1000, 20)

Loading Default Credit Card Dataset
Shape: (30000, 26)
Gender: 0(F)=18112, 1(M)=11888
  Final shape: (30000, 22)
  Sensitive groups: {0: 18112, 1: 11888}
✓ credit: (30000, 22)

Loading Law School (LSAC) Dataset
Shape: (18692, 14)
Gender: 0(F)=8142, 1(M)=10550
  Final shape: (18692, 10)
  Sensitive groups: {0: 8142, 1: 10550}
✓ law: (18692, 10)
Adult (race) | 0(NonWhite)=4229, 1(White)=25933
  Final shape: (30162, 12)
  Sensitive groups: {0: 4229, 1: 25933}
✓ adult_race: (30162, 12)
Adult 

## Cell 6: Algorithms

In [13]:
# =============================================================================
# SECTION 6: CLUSTERING ALGORITHMS (unchanged from original)
# =============================================================================

def standard_kmeans(X, k, random_state=42):
    km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    return km.fit_predict(X), km.cluster_centers_


def create_fairlets(X, sensitive_attr, p, q):
    n = len(X)
    g0, g1 = np.where(sensitive_attr == 0)[0], np.where(sensitive_attr == 1)[0]
    fairlets, fairlet_labels = [], np.full(n, -1)
    used0, used1, fid = set(), set(), 0
    while len(used0)+p <= len(g0) and len(used1)+q <= len(g1):
        s0 = np.random.choice([i for i in g0 if i not in used0], p, replace=False)
        s1 = np.random.choice([i for i in g1 if i not in used1], q, replace=False)
        members = np.concatenate([s0, s1])
        fairlets.append(members)
        fairlet_labels[members] = fid
        used0.update(s0); used1.update(s1); fid += 1
    rem = np.where(fairlet_labels == -1)[0]
    if len(rem):
        fairlets.append(rem); fairlet_labels[rem] = fid
    return fairlets, fairlet_labels


def fairlet_clustering(X, sensitive_attr, k, p=1, q=1):
    fairlets, _ = create_fairlets(X, sensitive_attr, p, q)
    centers_f = np.array([np.mean(X[f], axis=0) for f in fairlets])
    fc_labels = KMeans(n_clusters=min(k, len(centers_f)), random_state=42, n_init=10).fit_predict(centers_f) \
        if len(centers_f) >= k else np.arange(len(centers_f))
    labels = np.zeros(len(X), dtype=int)
    for i, cl in enumerate(fc_labels):
        labels[fairlets[i]] = cl
    centers = np.array([np.mean(X[labels == i], axis=0) if (labels == i).any()
                         else np.zeros(X.shape[1]) for i in range(k)])
    return labels, centers


def bfkm_clustering(X, sensitive_attr, k, max_iter=100):
    centers = X[np.random.choice(len(X), k, replace=False)].copy()
    for _ in range(max_iter):
        old = centers.copy()
        dists = cdist(X, centers)
        labels = np.argmin(dists, axis=1)
        t0, t1 = (sensitive_attr == 0).sum(), (sensitive_attr == 1).sum()
        for c in range(k):
            mask = labels == c
            s = sensitive_attr[mask]
            n0, n1 = (s == 0).sum(), (s == 1).sum()
            if t0 > 0 and t1 > 0:
                ratio_exp = t1 / t0
                ratio_act = n1 / max(n0, 1)
                if abs(ratio_act - ratio_exp) > 0.3:
                    idxs = np.where(mask)[0]
                    grp = 1 if ratio_act > ratio_exp else 0
                    g_idxs = idxs[sensitive_attr[idxs] == grp]
                    if len(g_idxs):
                        far = g_idxs[np.argmax(dists[g_idxs, c])]
                        sc = np.argsort(dists[far])
                        labels[far] = sc[1] if sc[1] != c else sc[2]
        for i in range(k):
            pts = X[labels == i]
            if len(pts): centers[i] = pts.mean(0)
        if np.allclose(centers, old, rtol=1e-4): break
    return labels, centers


class OptimizedFairCentroid:
    def __init__(self, n_clusters=6, max_iter=50, min_iter=10,
                 tol=0.005, random_state=0, candidate_fraction=0.05):
        self.n_clusters = n_clusters
        self.max_iter = max_iter; self.min_iter = min_iter
        self.tol = tol; self.random_state = random_state
        self.candidate_fraction = candidate_fraction
        self.labels_ = self.cluster_centers_ = self.inertia_ = None; self.n_iter_ = 0

    def _compute_cluster_fairness_fast(self, X_c, S_c, centroid):
        if len(X_c) == 0: return 0.0
        uq = np.unique(S_c)
        if len(uq) < 2: return 0.0
        d2 = np.sum((X_c - centroid)**2, axis=1)
        gm = [d2[S_c == g].mean() for g in uq if (S_c == g).sum() > 0]
        return max(gm) - min(gm) if len(gm) >= 2 else 0.0

    def _precompute_cluster_stats(self, X, labels, S, centers):
        stats = {}
        for c in range(self.n_clusters):
            mask = labels == c; n_c = mask.sum()
            if n_c == 0:
                stats[c] = {'n': 0, 'centroid': centers[c], 'fairness': 0.0, 'X': np.array([]), 'S': np.array([])}
            else:
                Xc, Sc = X[mask], S[mask]
                stats[c] = {'n': n_c, 'centroid': centers[c],
                            'fairness': self._compute_cluster_fairness_fast(Xc, Sc, centers[c]),
                            'X': Xc, 'S': Sc}
        return stats

    def _find_candidates(self, X, labels, centers, n_candidates):
        n = len(X)
        nc = max(1, int(n * self.candidate_fraction))
        dists = np.array([np.linalg.norm(X[i] - centers[labels[i]]) for i in range(n)])
        return np.argsort(-dists)[:nc]

    def fit(self, X, sensitive_attr):
        S = np.array(sensitive_attr if not hasattr(sensitive_attr, 'values') else sensitive_attr.values)
        km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        self.labels_ = km.fit_predict(X)
        centers = km.cluster_centers_.copy()
        for it in range(self.max_iter):
            stats = self._precompute_cluster_stats(X, self.labels_, S, centers)
            cands = self._find_candidates(X, self.labels_, centers, int(len(X)*self.candidate_fraction))
            improved = False
            for idx in cands:
                cur = self.labels_[idx]
                cur_fair = stats[cur]['fairness']
                best_c, best_delta = cur, 0
                for c2 in range(self.n_clusters):
                    if c2 == cur: continue
                    new_labels = self.labels_.copy()
                    new_labels[idx] = c2
                    new_centers = centers.copy()
                    for ci in [cur, c2]:
                        pts = X[new_labels == ci]
                        if len(pts): new_centers[ci] = pts.mean(0)
                    new_stats = self._precompute_cluster_stats(X, new_labels, S, new_centers)
                    delta = (stats[cur]['fairness'] + stats[c2]['fairness']) - \
                            (new_stats[cur]['fairness'] + new_stats[c2]['fairness'])
                    if delta > best_delta:
                        best_delta, best_c = delta, c2
                if best_c != cur:
                    self.labels_[idx] = best_c
                    for ci in [cur, best_c]:
                        pts = X[self.labels_ == ci]
                        if len(pts): centers[ci] = pts.mean(0)
                    stats = self._precompute_cluster_stats(X, self.labels_, S, centers)
                    improved = True
            if it >= self.min_iter and not improved: break
        self.cluster_centers_ = centers
        self.n_iter_ = it + 1
        return self


def fair_centroid_clustering(X, sensitive_attr, k, max_iter=20, min_iter=5, random_state=0):
    model = OptimizedFairCentroid(n_clusters=k, max_iter=max_iter,
                                   min_iter=min_iter, random_state=random_state)
    model.fit(X, sensitive_attr)
    return model.labels_, model.cluster_centers_

class PostProcessingNFP:
    def __init__(self, n_clusters=6, balance_tolerance=0.05, max_iter=30, random_state=0):
        self.n_clusters = n_clusters; self.balance_tolerance = balance_tolerance
        self.max_iter = max_iter; self.random_state = random_state
        self.labels_ = self.cluster_centers_ = None

    def fit(self, X, sensitive_attr):
        S = np.array(sensitive_attr)
        km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        self.labels_ = km.fit_predict(X)
        centers = km.cluster_centers_.copy()
        t0, t1 = (S == 0).sum(), (S == 1).sum()
        global_ratio = t1 / max(t0, 1)
        for _ in range(self.max_iter):
            changed = False
            ratios = []
            for c in range(self.n_clusters):
                mask = self.labels_ == c
                s = S[mask]
                ratios.append((s == 1).sum() / max((s == 0).sum(), 1))
            max_c, min_c = np.argmax(ratios), np.argmin(ratios)
            if abs(ratios[max_c] - global_ratio) <= self.balance_tolerance: break
            mask_max = np.where((self.labels_ == max_c) & (S == 1))[0]
            if len(mask_max) == 0: break
            dists_to_min = np.linalg.norm(X[mask_max] - centers[min_c], axis=1)
            nearest = mask_max[np.argmin(dists_to_min)]
            self.labels_[nearest] = min_c
            for ci in [max_c, min_c]:
                pts = X[self.labels_ == ci]
                if len(pts): centers[ci] = pts.mean(0)
            changed = True
            if not changed: break
        self.cluster_centers_ = centers
        return self


class PostProcessingGini:
    def __init__(self, n_clusters=6, k_neighbors=10, balance_tolerance=0.05, max_iter=30, random_state=0):
        self.n_clusters = n_clusters; self.k_neighbors = k_neighbors
        self.balance_tolerance = balance_tolerance; self.max_iter = max_iter
        self.random_state = random_state; self.labels_ = self.cluster_centers_ = None

    def _compute_gini_coefficient(self, group_counts):
        if sum(group_counts.values()) == 0: return 0
        total = sum(group_counts.values())
        probs = [v/total for v in group_counts.values()]
        return 1 - sum(p**2 for p in probs)

    def _compute_overall_gini(self, labels, S, k):
        ginis = []
        for c in range(k):
            mask = labels == c
            if mask.sum() == 0: continue
            s = S[mask]
            counts = {g: (s == g).sum() for g in np.unique(S)}
            ginis.append(self._compute_gini_coefficient(counts))
        return np.mean(ginis) if ginis else 0

    def fit(self, X, sensitive_attr):
        S = np.array(sensitive_attr)
        km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        self.labels_ = km.fit_predict(X)
        centers = km.cluster_centers_.copy()
        t0, t1 = (S == 0).sum(), (S == 1).sum()
        global_ratio = t1 / max(t0, 1)
        for _ in range(self.max_iter):
            ratios = [(S[(self.labels_ == c) & True] == 1).sum() /
                      max((S[self.labels_ == c] == 0).sum(), 1) for c in range(self.n_clusters)]
            max_c, min_c = np.argmax(ratios), np.argmin(ratios)
            if abs(ratios[max_c] - global_ratio) <= self.balance_tolerance: break
            # Gini-guided selection
            cands = np.where((self.labels_ == max_c) & (S == 1))[0]
            if len(cands) == 0: break
            ginis = []
            for idx in cands:
                tmp = self.labels_.copy(); tmp[idx] = min_c
                ginis.append(self._compute_overall_gini(tmp, S, self.n_clusters))
            best = cands[np.argmax(ginis)]
            self.labels_[best] = min_c
            for ci in [max_c, min_c]:
                pts = X[self.labels_ == ci]
                if len(pts): centers[ci] = pts.mean(0)
        self.cluster_centers_ = centers
        return self


def postprocessing_clustering_nfp(X, sensitive_attr, k, max_iter=30, random_state=0):
    m = PostProcessingNFP(n_clusters=k, max_iter=max_iter, random_state=random_state)
    m.fit(X, sensitive_attr)
    return m.labels_, m.cluster_centers_


def postprocessing_clustering_gini(X, sensitive_attr, k, max_iter=30, random_state=0):
    m = PostProcessingGini(n_clusters=k, max_iter=max_iter, random_state=random_state)
    m.fit(X, sensitive_attr)
    return m.labels_, m.cluster_centers_
    

def rawlsian_kmeans(X, sensitive_attr, k, n_runs=30, delta=None, random_state=0):
    Xd = X.toarray() if hasattr(X, 'toarray') else X
    S = sensitive_attr.values if hasattr(sensitive_attr, 'values') else np.array(sensitive_attr)
    if delta is None: delta = np.sqrt(Xd.shape[1])
    best_score, best_labels, best_centers = -np.inf, None, None
    for seed in range(n_runs):
        km = KMeans(n_clusters=k, random_state=random_state+seed, n_init=1)
        labels = km.fit_predict(Xd); centers = km.cluster_centers_
        utils = np.maximum(0, 1 - np.linalg.norm(Xd - centers[labels], axis=1) / delta)
        gu = [utils[S == g].mean() for g in np.unique(S) if (S == g).sum() > 0]
        score = min(gu) if gu else 0
        if score > best_score:
            best_score, best_labels, best_centers = score, labels.copy(), centers.copy()
    return best_labels, best_centers

## Cell 7: Metrics

In [15]:
# =============================================================================
# SECTION 7: METRICS (unchanged from original)
# =============================================================================

def calculate_quality_metrics(X, labels):
    k = len(np.unique(labels))
    centers = np.array([X[labels==i].mean(0) if (labels==i).any() else np.zeros(X.shape[1])
                         for i in range(k)])
    inertia = sum(np.sum((X[labels==i]-centers[i])**2) for i in range(k))
    sil = silhouette_score(X, labels) if 1 < k < len(X) else 0
    ch  = calinski_harabasz_score(X, labels) if 1 < k < len(X) else 0
    db  = davies_bouldin_score(X, labels) if 1 < k < len(X) else 0
    overall_mean = X.mean(0)
    wcss = sum(np.sum((X[labels==i]-centers[i])**2) for i in range(k))
    bcss = sum(len(X[labels==i])*np.sum((centers[i]-overall_mean)**2) for i in range(k))
    tss = np.sum((X-overall_mean)**2)
    vr = bcss/wcss if wcss > 0 else 0
    try:
        icd = np.mean(pdist(centers)) if k > 1 else 0
    except: icd = 0
    intra = [np.mean(pdist(X[labels==i])) for i in range(k) if (labels==i).sum() > 1]
    intra_d = np.mean(intra) if intra else 0
    sep = icd/intra_d if intra_d > 0 else 0
    try:
        inter_d = np.min(cdist(centers, centers)+np.eye(k)*1e9) if k > 1 else 0
        max_intra = max([np.max(pdist(X[labels==i])) for i in range(k)
                         if (labels==i).sum() > 1] or [1])
        dunn = inter_d/max_intra
    except: dunn = 0
    return {'Inertia': inertia, 'Silhouette': sil, 'Calinski_Harabasz': ch,
            'Davies_Bouldin': db, 'Dunn_Index': dunn, 'WCSS': wcss, 'BCSS': bcss,
            'TSS': tss, 'Variance_Ratio': vr, 'Inter_Cluster_Distance': icd,
            'Intra_Cluster_Distance': intra_d, 'Separation_Index': sep}


def calculate_balance(labels, S):
    balances = []
    for c in np.unique(labels):
        sc = S[labels==c]
        n0, n1 = (sc==0).sum(), (sc==1).sum()
        if n0 > 0 and n1 > 0:
            balances.append(min(n0,n1)/max(n0,n1))
    return np.mean(balances) if balances else 0.0


def calculate_spd(labels, S):
    t0, t1 = (S==0).sum(), (S==1).sum()
    vals = []
    for c in np.unique(labels):
        mask = labels==c
        p0 = (S[mask]==0).sum()/t0 if t0 > 0 else 0
        p1 = (S[mask]==1).sum()/t1 if t1 > 0 else 0
        vals.append(abs(p0-p1))
    return np.mean(vals)


def calculate_disparate_impact(labels, S):
    t0, t1 = (S==0).sum(), (S==1).sum()
    vals = []
    for c in np.unique(labels):
        mask = labels==c
        p0 = (S[mask]==0).sum()/t0 if t0 > 0 else 0
        p1 = (S[mask]==1).sum()/t1 if t1 > 0 else 0
        if p0 > 0 and p1 > 0:
            vals.append(min(p0,p1)/max(p0,p1))
    return np.mean(vals) if vals else 0.0


def calculate_entropy_metric(labels, S):
    entropies = []
    for c in np.unique(labels):
        sc = S[labels==c]
        n0, n1 = (sc==0).sum(), (sc==1).sum()
        if n0 > 0 and n1 > 0:
            p = np.array([n0, n1]) / len(sc)
            entropies.append(entropy(p, base=2))
    return np.mean(entropies) if entropies else 0.0

## Cell 8: Fairlet k-Sensitivity Sweep

In [20]:
import time
import numpy as np

K_RANGE = range(2, 11)

def run_fairlet_at_k(X, sensitive, k):
    np.random.seed(0)
    try:
        labels, _ = fairlet_clustering(X, sensitive, k)
        q = calculate_quality_metrics(X, labels)
        return {
            'k':          k,
            'Silhouette': q['Silhouette'],
            'Balance':    calculate_balance(labels, sensitive),
            'SPD':        calculate_spd(labels, sensitive),
            'status':     'ok',
        }
    except Exception as e:
        return {
            'k': k, 'Silhouette': np.nan,
            'Balance': np.nan, 'SPD': np.nan,
            'status': str(e)
        }

sweep_fairlet = {}

print("\n" + "="*70)
print("k-SENSITIVITY SWEEP — FAIRLET ONLY (k=2-10, full data)")
print("="*70)

for ds in dataset_names:
    X   = datasets[ds]['X']
    sen = datasets[ds]['sensitive']
    n   = X.shape[0]

    print(f"\n{ds.upper()} (n={n}):")

    t0  = time.perf_counter()
    res = [run_fairlet_at_k(X, sen, k) for k in K_RANGE]
    elapsed = time.perf_counter() - t0

    sweep_fairlet[ds] = res
    sils = [r['Silhouette'] for r in res if not np.isnan(r['Silhouette'])]
    if sils:
        print(f"  Fairlet  Sil=[{min(sils):.3f},{max(sils):.3f}]  [{elapsed:.1f}s]")
    else:
        print(f"  Fairlet  Sil=[ALL NaN]  [{elapsed:.1f}s]")
        for r2 in res:
            if r2['status'] != 'ok':
                print(f"    k={r2['k']}: {r2['status']}")

with open('k_sensitivity_fairlet.pkl', 'wb') as f:
    pickle.dump({
        'sweep_fairlet': sweep_fairlet,
        'datasets': dataset_names,
        'k_range': list(K_RANGE),
    }, f)
print("\n✓ Saved: k_sensitivity_fairlet.pkl")


k-SENSITIVITY SWEEP — FAIRLET ONLY (k=2-10, full data)

ADULT (n=30162):
  Fairlet  Sil=[-0.061,0.083]  [622.1s]

COMPAS (n=9395):
  Fairlet  Sil=[-0.124,0.015]  [52.0s]

GERMAN (n=1000):
  Fairlet  Sil=[-0.060,0.013]  [2.0s]

CREDIT (n=30000):
  Fairlet  Sil=[-0.051,0.036]  [726.9s]

LAW (n=18692):
  Fairlet  Sil=[-0.081,0.099]  [305.8s]

ADULT_RACE (n=30162):
  Fairlet  Sil=[-0.114,0.004]  [426.4s]

ADULT_COMBINED (n=30162):
  Fairlet  Sil=[-0.135,0.148]  [217.4s]

COMPAS_RACE (n=7931):
  Fairlet  Sil=[-0.063,0.054]  [62.6s]

COMPAS_COMBINED (n=7931):
  Fairlet  Sil=[-0.149,-0.011]  [25.5s]

LAW_RACE (n=18692):
  Fairlet  Sil=[-0.083,0.112]  [114.9s]

LAW_COMBINED (n=18692):
  Fairlet  Sil=[0.018,0.198]  [75.6s]

DIABETES_GENDER (n=99492):
  Fairlet  Sil=[-0.011,0.035]  [10871.7s]

DIABETES_RACE (n=95309):
  Fairlet  Sil=[-0.069,0.012]  [6707.2s]

DIABETES_COMBINED (n=95309):
  Fairlet  Sil=[-0.108,-0.012]  [5891.0s]

DUTCH_GENDER (n=60420):
  Fairlet  Sil=[-0.033,0.257]  [3555.3s]


## Cell 9: Merge Fairlet into Main k_sensitivity.pkl

In [30]:
# =============================================================================
# Merge Fairlet results into existing k_sensitivity.pkl
# Run this AFTER the main k-sensitivity sweep has completed
# =============================================================================

# Load existing main sweep
with open('k_sensitivity.pkl', 'rb') as f:
    main = pickle.load(f)

sweep = main['sweep']

# Add fairlet to each dataset
for ds in sweep_fairlet:
    if ds in sweep:
        sweep[ds]['fairlet'] = sweep_fairlet[ds]
        print(f"✓ Added Fairlet to {ds}")
    else:
        print(f"✗ {ds} not found in main sweep — skipping")

# Update methods list
methods_updated = ['kmeans','fairlet','bfkm','fair_centroid',
                   'postprocessing_nfp','postprocessing_gini','rawlsian']

# Save merged pkl
with open('k_sensitivity.pkl', 'wb') as f:
    pickle.dump({
        'sweep':    sweep,
        'datasets': main['datasets'],
        'methods':  methods_updated,
        'k_range':  main['k_range'],
    }, f)

print("\n✓ Saved: k_sensitivity.pkl (with Fairlet added)")
print(f"  Methods now: {methods_updated}")

# Verify
with open('k_sensitivity.pkl', 'rb') as f:
    check = pickle.load(f)
first_ds = list(check['sweep'].keys())[0]
print(f"\nVerification — {first_ds} methods: {list(check['sweep'][first_ds].keys())}")

✓ Added Fairlet to adult
✓ Added Fairlet to compas
✓ Added Fairlet to german
✓ Added Fairlet to credit
✓ Added Fairlet to law
✓ Added Fairlet to adult_race
✓ Added Fairlet to adult_combined
✓ Added Fairlet to compas_race
✓ Added Fairlet to compas_combined
✓ Added Fairlet to law_race
✓ Added Fairlet to law_combined
✓ Added Fairlet to diabetes_gender
✓ Added Fairlet to diabetes_race
✓ Added Fairlet to diabetes_combined
✓ Added Fairlet to dutch_gender
✓ Added Fairlet to meps_gender
✓ Added Fairlet to meps_race
✓ Added Fairlet to meps_combined

✓ Saved: k_sensitivity.pkl (with Fairlet added)
  Methods now: ['kmeans', 'fairlet', 'bfkm', 'fair_centroid', 'postprocessing_nfp', 'postprocessing_gini', 'rawlsian']

Verification — adult methods: ['kmeans', 'bfkm', 'fair_centroid', 'postprocessing_nfp', 'postprocessing_gini', 'rawlsian', 'fairlet']
